# Padding
We wish to build the real function

$$f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x)=\sum_{k\in{\mathbb{Z}}}\,c[k]\,\varphi(x-k).$$
There, the sequence $c$ of coefficients is used to parameterize the function $f$ and gives us a good many degrees of freedom to shape it to our taste. Moreover, the basis $\varphi$ is assumed to have the desirable technical property of being a Riesz basis. Among other things, this ensures that the infinite sum found in the construction of $f$ is always well-behaved.

The adaptability of $c$ makes it a tool of choice to represent sampled data as the continuously defined function $f.$ Given a sequence $y$ of regularly indexed samples $y[q]$ for $q\in{\mathbb{Z}},$ the so-called interpolation condition leads to a procedure that yields a sequence $c$ such that $f(q)=y[q].$ Fortunately, the assumption of a Riesz basis guarantees the existence of $c$ for any $y.$ Furthermore, in case $\varphi$ is a B-spline, there exist very efficient algorithms to get $c$ out of $y.$

Unfortunately, the theoretical derivations of the algorithms rely on $y$ being a *sequence*, which means that infinitely many samples are required. Now, one never has access to a sequence of samples in practice, only to a finite-dimensional *vector* ${\mathbf{y}}\in{\mathbb{R}}^{K}$ of samples. To take advantage of the theoretical derivations of the efficient algorithms, it is thus an unavoidable necessity that a procedure be engineered that converts the vector ${\mathbf{y}}$ into the sequence $y.$ We call padding the operation that consists in the engineering of the subsequences $\left(y[k]\right)_{k\in{\mathbb{Z}}_{<0}}$ to the left and $\left(y[k]\right)_{k\in{\mathbb{Z}}_{\geq K}}$ to the right of the provided $\left(y[k]\right)_{k=0}^{K-1}$.

The engineering of ${\mathbf{y}}\mapsto y$ is application-dependent. For instance, ${\mathbf{y}}$ could represent angular data, in which case one would have to cope with angular wrapping. Or, ${\mathbf{y}}$ could represent intensity data, in which case one would have to discourage negative intensities in the continuously defined function $f$ being constructed through the steps ${\mathbf{y}}\mapsto y\mapsto c\mapsto f.$ Or, one could pretend that all unobserved samples do vanish and take the special value $0.$ Or, one could assume that the first observed sample has indeed the same value as all unobserved samples that came before, while the last observed sample has the same value as all unobserved samples that folllow.

## Unicity
In the sequel, we let $\varphi$ be a polynomial B-spline of nonnegative integer degree $n,$ which is a real function $\beta^{n}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto\beta^{n}(x).$ For any degree $n\geq2$ and for $m\in[1\ldots\left\lfloor n/2\right\rfloor],$ it is known that there exist real, negative numbers $z_{n,m}$ in the open interval $(-1,0)$ that satisfy the relation

$$\frac{1}{\sum_{k\in{\mathbb{Z}}}\,\beta^{n}(k)\,z_{n,m}^{-k}}\not\in{\mathbb{C}}.$$ These numbers are called the poles of the spline and are such that $\sum_{k\in{\mathbb{Z}}}\,\beta^{n}(k)\,z_{n,m}^{-k}=0.$ Because B-splines are even-symmetric, the pole reciprocals $z_{n,m}^{-1}\in{\mathbb{R}}_{<-1}$ satisfy the same relations.

Suppose now we have identified a sequence $c$ that verifies the interpolation condition $y[q]=\sum_{k\in{\mathbb{Z}}}\,c[k]\,\beta^{n}(q-k).$ Then, the sequence $c'[k]=c[k]+\sum_{m=1}^{\left\lfloor n/2\right\rfloor}\,\left(\lambda_{n,m}^{-}\,z^{-k}+\lambda_{n,m}^{+}\,z^{k}\right)$ is also such that $y[q]=\sum_{k\in{\mathbb{Z}}}\,c'[k]\,\beta^{n}(q-k),$ for any choice of the constants $\lambda_{n,m}^{-},\lambda_{n,m}^{+}\in{\mathbb{R}}.$ One concludes that additional requirements are needed to make the interpolation task well-defined.

## Available Paddings
The unobserved samples are, well, unobserved. Consequently, every strategy that assigns specific values to them is valid, but some are less practical than others. The ``splinekit`` library deals with paddings of low complexity; in particular, we focus on some for which the overall organization of $y,$ $c,$ and $f$ can be made consistent and solve the unicity issue. The seven paddings being considered are

*   Periodic
*   Narrow Mirror
*   Wide Mirror
*   Anti-Mirror
*   Nega-Periodic
*   Nega-Narrow Mirror
*   Nega-Wide Mirror

Except for the anti-mirror padding, all forms are openly periodic over $y,$ albeit the length of a period may differ from the number $K$ of observed samples. We arbitrarily impose that $c$ follows the same periodicity than that of $y,$ which implies that $f$ is likewise periodic, too. Taking $c$ to be periodic, the trivial choice $\lambda_{n,m}^{-}=0$ and $\lambda_{n,m}^{+}=0$ is the only one that makes $c'$ periodic too, with $c'=c.$ Unicity is thus achieved. 

We illustrate now visually the effect of the various paddings on random splines of a specified degree. The data samples are represented with circles. The first sample, as well as its replicates when the padding is globally periodic, is indicated by a red stem line. The portion of curve in thick green is tied to the observed data, with a number of samples that can be specified. The portion of curve in thick blue is the complement to a full period.

In [ ]:
# Load the required libraries.
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_degree = 5 # Maximal spline degree
max_samples = 12 # Maximal support of the observed data samples
# Random periodic cubic spline
k0 = 6 # Initial number of observed samples
s0 = sk.PeriodicSpline1D.from_spline_coeff(np.random.standard_normal(k0), degree = 3)

# Plot
def update_plot (
    degree = 3,
    samples = 6,
    padding = 0,
    display = 2
):
    global k0
    global s0

    # Update of the degree
    if s0.degree != degree:
        s0 = s0.projected(degree = degree)

    # Update of the number of samples
    f0 = s0.get_samples(0, support_length = k0)
    if k0 < samples:
        f0 = np.append(f0, np.random.standard_normal(samples - len(f0)))
    else:
        f0 = f0[ : samples]
    k0 = len(f0) # Number of observed samples

    highlight = sk.interval.Empty() # Support of the observed data
    downlight = sk.interval.Empty() # Complementary support
    plt_domain = sk.interval.Empty() # Support over three periods
    f = f0

    if 0 == padding: # Periodic
        plt_domain = sk.interval.ClosedOpen((-k0 - 0.5, 2 * k0 + 0.5))
        highlight = sk.interval.ClosedOpen((0, k0))
    elif 1 == padding: # Narrow Mirror
        f = np.zeros(max(1, 2 * k0 - 2), dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_n(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((0, max(1, k0 - 1)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f)))
    elif 2 == padding: # Wide Mirror
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_w(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, k0 - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 0.5))
    elif 3 == padding: # Anti-Mirror
        f = np.zeros(max_degree + 1 + 3 * (2 * k0 - 2) + max_degree + 1, dtype = float)
        delay = max_degree + 1 + 2 * k0 - 2
        for k in range(len(f)):
            f[k] = sk.pad_a(f0, at = k - delay)
        f = np.roll(f, -delay)
        plt_domain = sk.interval.ClosedOpen((-2 * k0 + 2 - 0.5, 4 * k0 - 4 + 0.5))
        highlight = sk.interval.ClosedOpen((0, max(1, k0 - 1)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, 2 * k0 - 2))
    elif 4 == padding: # Nega-Periodic
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_np(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((0, k0))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f)))
    elif 5 == padding: # Nega-Narrow Mirror
        f = np.zeros(2 * k0 + 2, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_nn(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-1, k0))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 1))
    elif 6 == padding: # Nega-Wide Mirror
        f = np.zeros(2 * k0, dtype = float)
        for k in range(len(f)):
            f[k] = sk.pad_nw(f0, at = k)
        plt_domain = sk.interval.ClosedOpen((-len(f) - 0.5, 2 * len(f) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, k0 - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(f) - 0.5))
    s0 = sk.PeriodicSpline1D.from_samples(f, degree = degree)

    (fig, ax) = plt.subplots()
    if 0 == display: # Show the observed samples
        s0.plot(
            (fig, ax),
            plotdomain = sk.interval.ClosedOpen((-0.5, k0 - 0.5)),
            line_fmt = "-C2",
            line_width = 3,
            plotpoints = 101,
            knot_marker = "None"
        )
    elif 1 == display: # Show one period
        # Determine the range of the plot
        s0.plot(
            (fig, ax),
            plotdomain = list(highlight | downlight)[0],
            plotpoints = 301,
            knot_marker = "None"
        )
        s0.plot(
            (fig, ax),
            plotdomain = highlight,
            line_fmt = "-C2",
            line_width = 3,
            plotpoints = 101,
            knot_marker = "None"
        )
        if 0 < downlight.diameter:
            s0.plot(
                (fig, ax),
                plotdomain = downlight,
                line_fmt = "-C0",
                line_width = 3,
                plotpoints = 101,
                knot_marker = "None"
            )
    elif 2 == display: # Show three periods
        s0.plot((fig, ax), plotdomain = plt_domain, plotpoints = 301, knot_marker = "None")
        s0.plot(
            (fig, ax),
            plotdomain = highlight,
            line_fmt = "-C2",
            line_width = 3,
            plotpoints = 101,
            knot_marker = "None"
        )
        if 0 < downlight.diameter:
            s0.plot(
                (fig, ax),
                plotdomain = downlight,
                line_fmt = "-C0",
                line_width = 3,
                plotpoints = 101,
                knot_marker = "None"
            )
    plt.show()

widgets.interactive(
    update_plot,
    degree = (0, max_degree),
    samples = (1, max_samples),
    padding = widgets.RadioButtons(
        options = [
            ("Periodic", 0),
            ("Narrow Mirror", 1),
            ("Wide Mirror", 2),
            ("Anti-Mirror", 3),
            ("Nega-Periodic", 4),
            ("Nega-Narrow Mirror", 5),
            ("Nega-Wide Mirror", 6)
        ],
        value = 0,
        description = "Padding:",
        disabled = False
    ),
    display = widgets.RadioButtons(
        options = [
            ("Observed Samples", 0),
            ("One Period", 1),
            ("Three Periods", 2)
        ],
        value = 2,
        description = "Displayed Support:",
        disabled = False
    )
)


## Periodic Padding
An easy, general-purpose padding approach is to engineer the sequence $y$ of samples as the straighforward periodized version of the vector ${\mathbf{y}}\in{\mathbb{R}}^{K}.$ We also request that the sequence $c$ of coefficients be $K$-periodic, too, for any basis $\varphi.$ Ultimately, this implies that the function $f$ is itself $K$-periodic. In summary, the relations being satisfied for any $k\in{\mathbb{Z}}$ and any $x\in{\mathbb{R}}$ are

$$\begin{eqnarray*}y[k]&=&y[k+K]\\c[k]&=&c[k+K]\\f(x)&=&f(x+K).\end{eqnarray*}$$

### Algorithmic Considerations
In the context of a periodic padding, there are three major algorithmic approaches to the solution of the interpolation constraint $f(q)=y[q]$ for $q\in[0\ldots K-1].$

*   Linear Algebra
*   Discrete Fourier
*   Recursive Filtering

The linear-algebra approach first establishes an explicit system of $K$ linear equations. The $q$-th equation of the system would be $y[q]=\sum_{k=0}^{K-1}\,\left(\sum_{p\in{\mathbb{Z}}}\,\varphi(q-p\,K-k)\right)\,c[k].$ Tools of linear algebra would then be deployed to solve the system in terms of the unknown variables $c[k].$ The overall computational cost is ${\mathcal{O}}(K^{3})$ when general solvers are used, and the cost would be ${\mathcal{O}}(K^{2})$ for Toeplitz systems. For periodic paddings, the system is circulant and the overall computational cost reduces to ${\mathcal{O}}(K\,\log K)$ with Fourier-based techniques to solve linear-algebra inversion problems.

The discrete-Fourier approach is best described concisely in matrix notations. Let ${\mathbf{F}}\in{\mathbb{C}}^{K\times K}$ be the discrete Fourier transform, with the $\nu$-th row and $q$-th column entry given by ${\mathrm{e}}^{-{\mathrm{j}}\,\left(\nu-1\right)\,\frac{2\,\uppi}{K}\,\left(q-1\right)}$. Let the vector ${\mathbf{c}}$ represent one period of the periodic sequence $c.$ Moreover, let ${\mathbf{\upvarphi}}=(\sum_{p\in{\mathbb{Z}}}\,\varphi(p\,K+q))_{q=0}^{K-1}$ be the data-independent vector that contains the samples (at the integers) of the periodized basis $\varphi.$ Then, one has that ${\mathbf{c}}={\mathbf{F}}^{-1}\,\left(\left({\mathbf{F}}\,{\mathbf{y}}\right)\oslash\left({\mathbf{F}}\,{\mathbf{\upvarphi}}\right)\right),$ where $\oslash$ is an element-wise division. In practice, the Fourier transformation and its inverse are implemented via the fast Fourier algorithm, in which case the overall computational cost is ${\mathcal{O}}(K\,\log K).$

The recursive-filtering approach is the one followed in the ``splinekit`` library. It requires that the basis $\varphi$ has a finite support, is even-symmetric, and that the poles of the reciprocal of the $z$-transform of its samples at the integers are real numbers. These properties are all satisfied by the polynomial B-splines of nonnegative integer degree $n\in{\mathbb{N}}+2.$ Start the algorithm by letting ${\mathbf{c}}={\mathbf{y}}$. Then, iteratively for every one of the poles $z_{n,m}\in(-1,0)$ indexed by $m\in[1\ldots\left\lfloor n/2\right\rfloor]$ and associated to the degree $n,$ apply the in-place recursive updates

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&\frac{1}{1-z_{n,m}^{K}}\,\left(c[0]+\sum_{k=1}^{K-1}\,z_{n,m}^{k}\,c[K-k]\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\frac{\left(1-z_{n,m}\right)^{2}}{1-z_{n,m}^{K}}\,\left(c[K-1]+\sum_{k=0}^{K-2}\,z_{n,m}^{k+1}\,c[k]\right)\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1].\end{array}\right.$$
The overall computational cost is now ${\mathcal{O}}(K\,\left\lfloor n/2\right\rfloor).$ In practice, further acceleration can be achieved if the sums that appear in the recursive-update equations are truncated at that index $k$ where the term $z_{n,m}^{k}$ becomes negligible.

## Experimental Performance
The following experiment establishes some simple statistics over the gain in speed achieved by the recursive approach over the fast-Fourier-based one.
*   In the first table, the number $K$ of observed data are powers of two, a situation that is very much to the advantage of the discrete Fourier methods. For each such length $K$ and for each spline degree, we synthesize $50$ random vectors ${\mathbf{y}}\in{\mathbb{Z}}^{K}$ and let both the Fourier approach and the recursive approach determine the spline coefficients on the same data. We time those computations and report by how many times the recursive approach is faster relatively to the Fourier approach.
*   In the second table, we repeat the experiment, with the difference that the $50$ lengths are now chosen randomly in some range.

The conclusion of the experiments is unequivocal: The recursive approach is substantially faster than the fast-Fourier approach, at all lengths (except $K=1$), ranges of lengths, and all degrees being investigated.

In [ ]:
# Load the required libraries.
from math import fsum
import numpy as np
import time

import splinekit as sk # This library

max_degree = 9 # Maximal degree of the piecewise-polynomial splines
octaves = 14 # Number of dyadic ranges
nb_experiments = 50 # Number of experiments within a range of lengths

# Initialize the generator of random numbers
rng = np.random.default_rng()

# FFT approach
def fft_samples_to_coeff_p (
    samples,
    degree
):
    k0 = len(samples)
    if 1 == k0:
        return samples.copy()
    # Discrete Fourier of the periodized B-spline
    rdftb = np.fft.rfft(np.fromiter(
        (
            fsum(
                sk.b_spline(k - p * k0, degree)
                for p in range(
                    int((k - 0.5 * (degree - 1.0)) // k0),
                    int((k + 0.5 * (degree + 1.0)) // k0) + 1
                )
            )
            for k in range(k0)
        ),
        dtype = float,
        count = k0
    ))
    # Discrete Fourier of the samples
    rdftsamples = np.fft.rfft(samples)
    # Inverse discrete Fourier of the ratio
    return np.fft.irfft(
        np.fromiter(
            (rdftsamples[nu] / rdftb[nu] for nu in range(k0 // 2 + 1)),
            dtype = complex,
            count = k0 // 2 + 1
        ),
        n = k0
    )

# Table of results for daydic data lengths
print()
print("Acceleration for Dyadic Periods")
print("===========================================================")
print("   Degree |     2     3     4     5     6     7     8     9")
print("Period    |")
print("----------+------------------------------------------------")
for v in range(octaves + 1):
    # Create nb_experiments data vectors with random values and dyadic lengths
    kp = np.array([2 ** v for _ in range(nb_experiments)], dtype = int)
    data = [rng.standard_normal(k0) for k0 in kp]
    performance = [0, 0]
    for degree in range(2, max_degree + 1):
        # FFT-based approach to determine periodic spline coefficients
        fft_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            fft_data[experiment] = fft_samples_to_coeff_p(samples, degree)
        end = time.perf_counter()
        fft_duration = end - start
        # Recursive-based approach to determine periodic spline coefficients
        rec_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            c = samples.copy()
            sk.samples_to_coeff_p(c, degree = degree)
            rec_data[experiment] = c
        end = time.perf_counter()
        rec_duration = end - start
        # Relative speed
        performance += [fft_duration / rec_duration]
    print("K = {0:5d} | {2:5.2f} {3:5.2f} {4:5.2f} {5:5.2f} {6:5.2f} {7:5.2f} {8:5.2f} {9:5.2f}".format(
        2 ** v,
        2 ** v,
        performance[2],
        performance[3],
        performance[4],
        performance[5],
        performance[6],
        performance[7],
        performance[8],
        performance[9]
    ))
print("===========================================================")

# Table of results for random data lengths
print()
print()
print("Acceleration for Random Periods")
print("====================================================================")
print("            Degree |     2     3     4     5     6     7     8     9")
print("Period Range       |")
print("-------------------+------------------------------------------------")
for v in range(octaves + 1):
    # Create nb_experiments data vectors with random values and random lengths
    kp = rng.integers(low = 2 ** v, high = 2 ** (v + 1), size = nb_experiments)
    data = [rng.standard_normal(k0) for k0 in kp]
    performance = [0, 0]
    for degree in range(2, max_degree + 1):
        # FFT-based approach to determine periodic spline coefficients
        fft_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            fft_data[experiment] = fft_samples_to_coeff_p(samples, degree)
        end = time.perf_counter()
        fft_duration = end - start
        # Recursive-based approach to determine periodic spline coefficients
        rec_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            c = samples.copy()
            sk.samples_to_coeff_p(c, degree = degree)
            rec_data[experiment] = c
        end = time.perf_counter()
        rec_duration = end - start
        # Relative speed
        performance += [fft_duration / rec_duration]
    print("{0:5d} <= K < {1:5d} | {2:5.2f} {3:5.2f} {4:5.2f} {5:5.2f} {6:5.2f} {7:5.2f} {8:5.2f} {9:5.2f}".format(
        2 ** v,
        2 ** (v + 1),
        performance[2],
        performance[3],
        performance[4],
        performance[5],
        performance[6],
        performance[7],
        performance[8],
        performance[9]
    ))
print("====================================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################


# Desktop computer of year 2021: ~25s



## Other Paddings
All forms of padding considered in the ``spline_padding`` module of the ``splinekit`` library involve some form of periodicity or pseudo-periodicity. An important benefit is that this leads to the unicity of the solution to the interpolation problem. While some paddings (*e.g.*, narrow-mirror and wide-mirror) have relevant practical benefits (*e.g.*, in image processing) and theoretical benefits (*e.g.*, in the development of wavelet algorithms), we focus on the periodic padding due to its simplicity. Without further ado, we list the properties of the remaining ones.

### Narrow Mirror
Assumptions $\forall k\in{\mathbb{Z}}$

$$\begin{eqnarray*}y[k]&=&y[-k]\\y[k+K-1]&=&y[K-1-k]\end{eqnarray*}$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$y[k]=y[k+2\,K-2]$$
$$\left\{\begin{array}{rcl}c[k]&=&c[-k]\\c[k+K-1]&=&c[K-1-k]\\c[k]&=&c[k+2\,K-2]\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x)&=&f(-x)\\f(x+K-1)&=&f(K-1-x)\\f(x)&=&f(x+2\,K-2)\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&\frac{1}{1-z_{n,m}^{2\,K-2}}\,\sum_{k=0}^{K-2}\,z_{n,m}^{k}\,\left(c[k]+z_{n,m}^{K-1}\,c[K-1-k]\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\frac{\left(1-z_{n,m}\right)^{2}}{1-z_{n,m}^{2}}\,\left(z_{n,m}\,c[K-2]+c[K-1]\right)\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$


### Wide Mirror
Assumptions $\forall k\in{\mathbb{Z}}$

$$\begin{eqnarray*}y[k]&=&y[-1-k]\\y[k+K]&=&y[K-1-k]\end{eqnarray*}$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$y[k]=y[k+2\,K]$$
$$\left\{\begin{array}{rcl}c[k]&=&c[-1-k]\\c[k+K]&=&c[K-1-k]\\c[k]&=&c[k+2\,K]\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x)&=&f(-1-x)\\f(x+K)&=&f(K-1-x)\\f(x)&=&f(x+2\,K)\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&c[0]+\frac{z_{n,m}}{1-z_{n,m}^{2\,K}}\,\sum_{k=0}^{K-1}\,z_{n,m}^{k}\,\left(c[k]+z_{n,m}^{K}\,c[K-1-k]\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\left(1-z_{n,m}\right)\,c[K-1]\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$


### Anti-Mirror
Assumptions $\forall k\in{\mathbb{Z}}$

$$\begin{eqnarray*}y[k]-y[0]&=&y[0]-y[-k]\\y[k+K-1]-y[K-1]&=&y[K-1]-y[K-1-k]\end{eqnarray*}$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$y[k]=y[k+2\,K-2]-2\,\left(y[K-1]-y[0]\right)$$
$$\left\{\begin{array}{rcl}c[k]-c[0]&=&c[0]-c[-k]\\c[k+K-1]-c[K-1]&=&c[K-1]-c[K-1-k]\\c[k]&=&c[k+2\,K-2]-2\,\left(c[K-1]-c[0]\right)\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x)-f(0)&=&f(0)-f(-x)\\f(x+K-1)-f(K-1)&=&f(K-1)-f(K-1-x)\\f(x)&=&f(x+2\,K-2)-2\,\left(f(K-1)-f(0)\right)\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&\frac{1}{1-z_{n,m}^{2\,K-2}}\,\left(\frac{1+z_{n,m}}{1-z_{n,m}}\,\left(c[0]-z_{n,m}^{K-1}\,c[K-1]\right)\right.\\&&\left.\mbox{}-\sum_{k=1}^{K-2}\,z_{n,m}^{k}\,\left(c[k]-z_{n,m}^{K-1}\,c[K-1-k]\right)\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&c[K-1]-z_{n,m}\,c[K-2]\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$


### Nega-Periodic
Assumptions $\forall k\in{\mathbb{Z}}$

$$y[k]=-y[k+K]$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$y[k]=y[k+2\,K]$$
$$\left\{\begin{array}{rcl}c[k]&=&-c[k+K]\\c[k]&=&c[k+2\,K]\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x)&=&-f(x+K)\\f(x)&=&f(x+2\,K)\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&c[0]-\frac{z_{n,m}}{1+z_{n,m}^{K}}\,\sum_{k=0}^{K-1}\,z_{n,m}^{K-1-k}\,c[k]\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\frac{1-z_{n,m}}{1+z_{n,m}}\,\left(\left(1+z_{n,m}^{2\,K}\right)\,c[K-1]-\frac{1}{1+z_{n,m}^{K}}\,\sum_{k=0}^{K-1}\,\left(z_{n,m}^{3\,K-1-k}+z_{n,m}^{k+1}\right)\,c[k]\right)\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$


### Nega-Narrow Mirror
Assumptions $\forall k\in{\mathbb{Z}}$

$$\begin{eqnarray*}y[k-1]&=&-y[-1-k]\\y[k+K+1]&=&-y[K-1-k]\end{eqnarray*}$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$\left\{\begin{array}{rcl}y[k]&=&y[k+2\,K+2]\\y[\left(K+1\right)\,k-1]&=&0\end{array}\right.$$
$$\left\{\begin{array}{rcl}c[k-1]&=&-c[-1-k]\\c[k+K+1]&=&-c[K-1-k]\\c[k]&=&c[k+2\,K+2]\\c[\left(K+1\right)\,k-1]&=&0\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x-1)&=&-f(-1-x)\\f(x+K+1)&=&-f(K-1-x)\\f(x)&=&f(x+2\,K+2)\\f(\left(K+1\right)\,k-1)&=&0\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&c[0]-\frac{z_{n,m}^{2}}{1-z_{n,m}^{2\,K+2}}\,\sum_{k=0}^{K-1}\,z_{n,m}^{k}\,\left(c[k]-z_{n,m}^{K+1}\,c[K-1-k]\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\left(1-z_{n,m}\right)^{2}\,c[K-1]\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$


### Nega-Wide Mirror
Assumptions $\forall k\in{\mathbb{Z}}$

$$\begin{eqnarray*}y[k]&=&-y[-1-k]\\y[k+K]&=&-y[K-1-k]\end{eqnarray*}$$
Consequences $\forall k\in{\mathbb{Z}},\forall x\in{\mathbb{R}}$

$$y[k]=y[k+2\,K]$$
$$\left\{\begin{array}{rcl}c[k]&=&-c[-1-k]\\c[k+K]&=&-c[K-1-k]\\c[k]&=&c[k+2\,K]\end{array}\right.$$
$$\left\{\begin{array}{rcl}f(x)&=&-f(-1-x)\\f(x+K)&=&-f(K-1-x)\\f(x)&=&f(x+2\,K)\end{array}\right.$$
Recursive in-Place Algorithm

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&c[0]-\frac{z_{n,m}}{1-z_{n,m}^{2\,K}}\,\sum_{k=0}^{K-1}\,z_{n,m}^{k}\,\left(c[k]-z_{n,m}^{K}\,c[K-1-k]\right)\\c[k]&\leftarrow&c[k]+z_{n,m}\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\frac{\left(1-z_{n,m}\right)^{2}}{1+z_{n,m}}\,c[K-1]\\c[K-1-k]&\leftarrow&z_{n,m}\,c[K-k]+\left(1-z_{n,m}\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1]\end{array}\right.$$
